# Installations

In [ ]:
%pip install kagglehub


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import kagglehub

In [ ]:
%pip install transformers datasets accelerate -q
%pip install huggingface_hub


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install bertopic -q


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from bertopic import BERTopic

In [ ]:
%pip install openai

In [ ]:
import openai
import json

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
import pickle
import os
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Install kaggle dataset

In [ ]:
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

import pandas as pd
true_dataset_path = path + '\True.csv'
fake_dataset_path = path + "\Fake.csv"

df_true = pd.read_csv(true_dataset_path)
df_fake = pd.read_csv(fake_dataset_path)

# Help functions

In [152]:
def clean_date(d):
  d = str(d).strip()
  if d.startswith('http') or d == '' or len(d) > 20 or d == 'nan':
    return np.nan
  try:
      return pd.to_datetime(d).strftime('%Y-%m-%d')
  except:
      return np.nan

In [153]:
def clean_text(text):
  text = str(text)
  text = re.sub(r'^[A-Z][a-z]+ [A-Z][a-z]+ says\s*', '', text)
  text = re.sub(r'Featured image.*$', '', text, flags=re.I)
  text = re.sub(r'Image source.*$', '', text, flags=re.I)
  text = re.sub(r'\s+', ' ', text).strip()
  text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
  text = re.sub(r'@\w+', '[MENTION]', text)
  text = re.sub(r'#\w+', '[TAG]', text)
  text = re.sub(r'\b\w+\.(?:com|org|net|gov|edu|co|uk|io)\b', '[DOMAIN]', text, flags=re.IGNORECASE)
  text = re.sub(r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b', '[DATE]', text)
  text = re.sub(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}\b', '[DATE]', text, flags=re.IGNORECASE)
  text = re.sub(r'\([^)]*(?:Reuters|AP|AFP|BBC|CNN|NBC|ABC|CBS|FOX)[^)]*\)', '[SOURCE]', text, flags=re.IGNORECASE)
  source_words = ['reuters', 'associated press', 'afp', 'bbc', 'cnn', 'bloomberg', 'wsj']
  for word in source_words:
      text = re.sub(rf'\b{word}\b', '', text, flags=re.IGNORECASE)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

# Data preprocesing

## Combine datasets

In [165]:
# Add label columns
df_true['label'] = 0 
df_fake['label'] = 1 

# Combine the datasets
df_true_fake = pd.concat([df_true, df_fake], ignore_index=True)

# Shuffle
df_true_fake = df_true_fake.sample(frac=1, random_state=42).reset_index(drop=True)


## Removing duplicates

In [166]:
print(len(df_true_fake))
df_true_fake = df_true_fake.drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
print(len(df_true_fake))

44898
39105


## Reconstructing data 

In [167]:
# Adding new columns with clean data
df_true_fake['date'] = df_true_fake['date'].apply(clean_date)
df_true_fake = df_true_fake.dropna(subset=['date']).reset_index(drop=True)

df_true_fake['text_clean'] = df_true_fake['text'].apply(clean_text)
df_true_fake['title_clean'] = df_true_fake['title'].apply(clean_text)

df_true_fake['content'] = df_true_fake['title_clean'] + "." + df_true_fake['text_clean']

# Removing columns 
columns_to_drop = ['text', 'title']
df_true_fake = df_true_fake.drop(columns=columns_to_drop)

# Bertopic and Topic mapping

In [ ]:
# Running Bertopic
print("\nRunning BERTopic...")
topic_model = BERTopic(language="english", verbose=True)
texts = df_true_fake['content'].fillna('').astype(str).tolist()
topics, probs = topic_model.fit_transform(texts)


print(f"Number of documents: {len(texts)}")
print(f"Number of topics found: {len(set(topics)) - (1 if -1 in topics else 0)}")

# Get topic probabilities for each document
topic_info = topic_model.get_document_info(texts)
print(f"Topic info shape: {topic_info.shape}")




Running BERTopic...


2026-01-10 18:47:40,733 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 1222/1222 [43:14<00:00,  2.12s/it]
2026-01-10 19:31:13,434 - BERTopic - Embedding - Completed ✓
2026-01-10 19:31:13,448 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-10 19:31:39,707 - BERTopic - Dimensionality - Completed ✓
2026-01-10 19:31:39,729 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-10 19:31:52,147 - BERTopic - Cluster - Completed ✓
2026-01-10 19:31:52,541 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-10 19:32:48,721 - BERTopic - Representation - Completed ✓


Number of documents: 39099
Number of topics found: 587
Topic info shape: (39099, 8)


  0%|          | 0/40 [00:48<?, ?it/s]


MemoryError: Unable to allocate 1.67 GiB for an array with shape (382890, 587) and data type float64

## Saving Bertopic to Hugging Face Hub

In [177]:
model_dir = "bert_fake_true"
topic_model.save(model_dir, save_embedding_model=True, save_ctfidf=True,
                      serialization="pytorch")

from huggingface_hub import upload_folder

if hasattr(topic_model.ctfidf_model, '_idf_diag'):
    import scipy.sparse as sp
    idf_diag = topic_model.ctfidf_model._idf_diag

    if sp.issparse(idf_diag):
        idf_values = idf_diag.diagonal()
    else:
        idf_values = idf_diag

    topic_model.ctfidf_model._idf_diag = idf_values

upload_folder(
    folder_path=model_dir,
    repo_id="rasm01/bert_fake_true",
    commit_message="Full BERTopic model upload with all files",
)

print("Loading is finished!")

ctfidf.bin:   0%|          | 0.00/23.4M [00:00<?, ?B/s]
ctfidf.bin:   0%|          | 16.4k/23.4M [00:00<15:29, 25.1kB/s]

ctfidf.bin:   1%|          | 197k/23.4M [00:00<01:11, 325kB/s]  
ctfidf.bin:   2%|▏         | 573k/23.4M [00:01<00:27, 829kB/s]
topic_embeddings.bin: 100%|██████████| 905k/905k [00:01<00:00, 595kB/s]
ctfidf.bin: 100%|██████████| 23.4M/23.4M [00:04<00:00, 4.78MB/s]
Upload 2 LFS files: 100%|██████████| 2/2 [00:05<00:00,  2.81s/it]


Loading is finished!


## Create topic mapping

In [ ]:

OPENAI_API_KEY = "_"

def create_simple_topic_mapping(topic_model, filename="topic_mapping.json"):
    """
    Create simple topic mapping using OpenAI API.
    
    Args:
        topic_model: Trained BERTopic model
        filename: Where to save the JSON mapping
    """
    
    # Initialize OpenAI
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    
    # Get all topic IDs 
    topic_ids = sorted(set(topic_model.topics_))
    topic_ids = [t for t in topic_ids if t != -1]
    
    print(f"Found {len(topic_ids)} topics")
    
    # Create mapping dictionary
    mapping = {}
    
    for topic_id in topic_ids:
        print(f"Naming topic {topic_id}...")
        
        # Get top keywords for this topic
        try:
            topic_words = topic_model.get_topic(topic_id)
            if topic_words:
                keywords = [word for word, _ in topic_words[:10]]
                keyword_text = ", ".join(keywords)
            else:
                keyword_text = "no keywords found"
        except:
            keyword_text = "error getting keywords"
        
        # Ask OpenAI for a simple name
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You create short, descriptive topic names (2-3 words)."},
                    {"role": "user", "content": f"Create a short topic name for texts about: {keyword_text}. Respond with just the name."}
                ],
                temperature=0.3,
                max_tokens=15
            )
            
            topic_name = response.choices[0].message.content.strip()
            topic_name = topic_name.strip('"\'')
            print(f"  -> {topic_name}")
            
        except Exception as e:
            print(f"  -> Error: {e}. Using fallback name.")
            topic_name = f"topic_{topic_id}"
        
        mapping[topic_id] = topic_name
        # Small delay to avoid rate limits
        import time
        time.sleep(0.5)
    
    # Add outliers topic
    mapping[-1] = "outliers"
    
    # Save to JSON file
    with open(filename, 'w') as f:
        json.dump(mapping, f, indent=2)
    
    print(f"\n Saved mapping to {filename}")
    return mapping



In [179]:

mapping = create_simple_topic_mapping(topic_model)

Found 587 topics
Naming topic 0...
  -> Healthcare Reform.
Naming topic 1...
  -> FBI Email Investigation
Naming topic 2...
  -> Corporate Tax Reform
Naming topic 3...
  -> Travel Ban Court Case
Naming topic 4...
  -> Berkeley Protests
Naming topic 5...
  -> Russian Election Hacking
Naming topic 6...
  -> Catalan Independence Crisis
Naming topic 7...
  -> Kurdish Politics
Naming topic 8...
  -> Police Shooting Incidents
Naming topic 9...
  -> Comey Investigation
Naming topic 10...
  -> Brexit Negotiations
Naming topic 11...
  -> Gun Control Measures
Naming topic 12...
  -> NFL Anthem Controversy
Naming topic 13...
  -> Diversity in Education
Naming topic 14...
  -> Venezuelan Politics
Naming topic 15...
  -> Flynn-Russia Contacts
Naming topic 16...
  -> Zimbabwean Politics
Naming topic 17...
  -> Hariri Resignation Crisis
Naming topic 18...
  -> SCOTUS Nominee Battle
Naming topic 19...
  -> Jerusalem Embassy Decision
Naming topic 20...
  -> Ryan's Republican Nomination
Naming topic 21.

# TF-IDF + Logistic Regression

In [184]:

# Sets up features and target
target_column = 'label'

X = df_true_fake['title_clean'].fillna('')
y = df_true_fake[target_column]

print(f"Samples: {len(X)}")
print(f"True: {(y == 0).sum()}, False: {(y == 1).sum()}")

#  Split data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
print("\nTraining TF-IDF...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


# LOGISTIC REGRESSION with hzperparamerer tuning
print("\nTraining Logistic Regression with hyperparameter tuning...")

# Simplified parameter grid
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_tfidf, y_train)
classifier = grid_search.best_estimator_

print(f"Best params: {grid_search.best_params_}")
print(f"CV accuracy: {grid_search.best_score_:.4f}")

# EVALUATION
print("\nEvaluating model...")

y_pred = classifier.predict(X_test_tfidf)
y_pred_proba = classifier.predict_proba(X_test_tfidf)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['False', 'True']))


Samples: 39099
True: 21197, False: 17902

Training TF-IDF...

Training Logistic Regression with hyperparameter tuning...


c:\Users\maria\Documents\VSE\coding\misinformation-trends-analyzer\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
30 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\maria\Documents\VSE\coding\misinformation-trends-analyzer\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\maria\Documents\VSE\coding\misinformation-trends-analyzer\.venv\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^

Best params: {'C': 10, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
CV accuracy: 0.9359

Evaluating model...
Test accuracy: 0.9394
Test ROC-AUC: 0.9846

Classification Report:
              precision    recall  f1-score   support

       False       0.94      0.95      0.94      4240
        True       0.94      0.93      0.93      3580

    accuracy                           0.94      7820
   macro avg       0.94      0.94      0.94      7820
weighted avg       0.94      0.94      0.94      7820



In [185]:
# FEATURE IMPORTANCE
if hasattr(classifier, 'coef_'):
    feature_names = vectorizer.get_feature_names_out()
    coefficients = classifier.coef_[0]
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefficients
    })
    
    top_true = feature_importance.nlargest(5, 'coefficient')
    top_false = feature_importance.nsmallest(5, 'coefficient')
    
    print("\nTop 5 TRUE features:")
    for _, row in top_true.iterrows():
        print(f"  {row['feature']}: {row['coefficient']:.4f}")
    
    print("\nTop 5 FALSE features:")
    for _, row in top_false.iterrows():
        print(f"  {row['feature']}: {row['coefficient']:.4f}")


Top 5 TRUE features:
  video: 29.5730
  hillary: 15.0367
  breaking: 14.3162
  gop: 13.6880
  just: 12.7887

Top 5 FALSE features:
  factbox: -10.8465
  says: -8.6216
  urges: -7.8751
  myanmar: -7.3109
  islamic state: -6.9211


In [186]:
# SAVE MODELS
model_dir = "fake_news_detector"
os.makedirs(model_dir, exist_ok=True)

with open(os.path.join(model_dir, "vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)

with open(os.path.join(model_dir, "classifier.pkl"), "wb") as f:
    pickle.dump(classifier, f)

print(f"\nModels saved to '{model_dir}/'")



print("\nModel ready for use.")


Models saved to 'fake_news_detector/'

Model ready for use.


In [187]:
# PREDICTION FUNCTION
def predict_news(text):
    if not text or text.strip() == "":
        return "Error: Empty text"
    
    text_tfidf = vectorizer.transform([text])
    prediction = classifier.predict(text_tfidf)[0]
    probability = classifier.predict_proba(text_tfidf)[0]
    
    label = "TRUE" if prediction == 0 else "FALSE"
    confidence = probability[prediction]
    
    return f"{label} ({confidence:.1%})"

In [188]:
print(predict_news('Trump is dead'))

FALSE (54.8%)


### Save log_reg an vectorizer to huggin face repo

In [189]:
model_dir = "log_reg_fake_true"
print(f"\nSaving model to '{model_dir}'...")

import pickle
import os

# Create directory
os.makedirs(model_dir, exist_ok=True)

# Save vectorizer
with open(os.path.join(model_dir, "vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)

# Save classifier
with open(os.path.join(model_dir, "classifier.pkl"), "wb") as f:
    pickle.dump(classifier, f)

print(f"✅ Model saved to '{model_dir}' folder")

# 3. Upload to Hugging Face
from huggingface_hub import upload_folder

print(f"\nUploading to Hugging Face...")
upload_folder(
    folder_path=model_dir,
    repo_id="rasm01/log_reg_fake_true",  # Your repo name
    commit_message=f"Logistic Regression model for fake news detection (Accuracy: {accuracy:.2%})",
)

print("✅ Model uploaded to Hugging Face!")
print(f"📦 Available at: https://huggingface.co/rasm01/log_reg_fake_true")


Saving model to 'log_reg_fake_true'...
✅ Model saved to 'log_reg_fake_true' folder

Uploading to Hugging Face...


classifier.pkl:   0%|          | 0.00/40.7k [00:00<?, ?B/s]
classifier.pkl:  40%|████      | 16.4k/40.7k [00:00<00:01, 22.0kB/s]

classifier.pkl: 100%|██████████| 40.7k/40.7k [00:01<00:00, 38.4kB/s]
vectorizer.pkl: 100%|██████████| 189k/189k [00:01<00:00, 125kB/s]
Upload 2 LFS files: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]


✅ Model uploaded to Hugging Face!
📦 Available at: https://huggingface.co/rasm01/log_reg_fake_true
